# Code Generation with LSTM and Transformer (GPT)

**Task:**
- Choose a programming language (Python)
- Download code samples from the Internet
- Build two networks: LSTM and Transformer (GPT)
- Train both on Python code
- Generate Python code with the trained models

## 1. Imports and parameters

In [ ]:
try:
    from silence_tensorflow import silence_tensorflow
    silence_tensorflow()
except ImportError:
    pass

import os
import re
import string
import urllib.request
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, losses

print("TensorFlow:", tf.__version__)

In [ ]:
# Common parameters
VOCAB_SIZE = 8000
MAX_LEN = 120
EMBEDDING_DIM = 128
BATCH_SIZE = 32
EPOCHS_LSTM = 8
EPOCHS_GPT = 5
VALIDATION_SPLIT = 0.15
SEED = 42

# LSTM
N_UNITS = 128

# Transformer (GPT)
KEY_DIM = 128
N_HEADS = 2
FEED_FORWARD_DIM = 256

## 2. Downloading Python code samples

We fetch Python code from public repositories (cpython standard library and small modules).

In [ ]:
# URLs of Python files (raw GitHub - cpython/Lib)
PYTHON_CODE_URLS = [
    "https://raw.githubusercontent.com/python/cpython/main/Lib/abc.py",
    "https://raw.githubusercontent.com/python/cpython/main/Lib/collections/__init__.py",
    "https://raw.githubusercontent.com/python/cpython/main/Lib/datetime.py",
    "https://raw.githubusercontent.com/python/cpython/main/Lib/functools.py",
    "https://raw.githubusercontent.com/python/cpython/main/Lib/heapq.py",
    "https://raw.githubusercontent.com/python/cpython/main/Lib/enum.py",
    "https://raw.githubusercontent.com/python/cpython/main/Lib/random.py",
    "https://raw.githubusercontent.com/python/cpython/main/Lib/re/__init__.py",
    "https://raw.githubusercontent.com/python/cpython/main/Lib/statistics.py",
    "https://raw.githubusercontent.com/python/cpython/main/Lib/types.py",
    "https://raw.githubusercontent.com/python/cpython/main/Lib/textwrap.py",
    "https://raw.githubusercontent.com/python/cpython/main/Lib/bisect.py",
    "https://raw.githubusercontent.com/python/cpython/main/Lib/copy.py",
    "https://raw.githubusercontent.com/python/cpython/main/Lib/keyword.py",
    "https://raw.githubusercontent.com/python/cpython/main/Lib/operator.py",
]

def download_python_code(urls, max_chars_per_file=50000):
    """Download Python file contents from the given URLs."""
    code_samples = []
    for url in urls:
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=10) as r:
                content = r.read().decode("utf-8", errors="replace")
                if len(content) > 100:
                    content = content[:max_chars_per_file]
                    code_samples.append(content)
        except Exception as e:
            print(f"Skip {url}: {e}")
    return code_samples

code_files = download_python_code(PYTHON_CODE_URLS)
print(f"Files downloaded: {len(code_files)}")
if code_files:
    print(f"Example (first 400 chars):\n{code_files[0][:400]}...")

In [ ]:
# If download failed (no network), use built-in examples
if len(code_files) < 3:
    print("Using built-in code examples (fallback).")
    code_files = [
        '''
def factorial(n):
    if n <= 1:
        return 1
    return n * factorial(n - 1)

def main():
    for i in range(10):
        print(i, factorial(i))
if __name__ == "__main__":
    main()
''',
        '''
class Stack:
    def __init__(self):
        self.items = []
    def push(self, x):
        self.items.append(x)
    def pop(self):
        return self.items.pop()
    def is_empty(self):
        return len(self.items) == 0
''',
        '''
import math
def is_prime(n):
    if n < 2:
        return False
    for i in range(2, int(math.sqrt(n)) + 1):
        if n % i == 0:
            return False
    return True
result = [x for x in range(100) if is_prime(x)]
print(result)
''',
    ]

# Split into reasonably sized "documents" (by lines)
def split_into_chunks(code_list, chunk_lines=25):
    out = []
    for code in code_list:
        lines = code.split("\n")
        for i in range(0, len(lines), chunk_lines):
            chunk = "\n".join(lines[i:i + chunk_lines])
            if len(chunk.strip()) > 50:
                out.append(chunk)
    return out

text_data = split_into_chunks(code_files, chunk_lines=30)
print(f"Number of sequences (chunks) for training: {len(text_data)}")

## 3. Preprocessing and tokenization

We pad punctuation so that symbols (parentheses, colons, etc.) are separate tokens.

In [ ]:
def pad_punctuation(s):
    s = re.sub(f"([{string.punctuation}, '\\n'])", r" \1 ", s)
    s = re.sub(" +", " ", s)
    return s.strip()

text_data = [pad_punctuation(t) for t in text_data]

text_ds = (
    tf.data.Dataset.from_tensor_slices(text_data)
    .batch(BATCH_SIZE)
    .shuffle(1000, seed=SEED)
)

vectorize_layer = layers.TextVectorization(
    standardize="lower",
    max_tokens=VOCAB_SIZE,
    output_mode="int",
    output_sequence_length=MAX_LEN + 1,
)
vectorize_layer.adapt(text_ds)
vocab = vectorize_layer.get_vocabulary()
print(f"Vocabulary size: {len(vocab)}")

In [ ]:
def prepare_inputs(text):
    text = tf.expand_dims(text, -1)
    tokenized = vectorize_layer(text)
    x = tokenized[:, :-1]
    y = tokenized[:, 1:]
    return x, y

train_ds = text_ds.map(prepare_inputs)
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)

## 4. LSTM model

In [ ]:
inputs = layers.Input(shape=(None,), dtype=tf.int32)
x = layers.Embedding(VOCAB_SIZE, EMBEDDING_DIM)(inputs)
x = layers.LSTM(N_UNITS, return_sequences=True)(x)
outputs = layers.Dense(VOCAB_SIZE, activation="softmax")(x)
lstm_model = models.Model(inputs=inputs, outputs=outputs)
lstm_model.compile(
    optimizer="adam",
    loss=losses.SparseCategoricalCrossentropy(),
)
lstm_model.summary()

In [ ]:
class TextGeneratorLSTM(callbacks.Callback):
    def __init__(self, index_to_word):
        self.index_to_word = list(index_to_word)
        self.word_to_index = {w: i for i, w in enumerate(self.index_to_word)}

    def _token_to_word(self, idx):
        if 0 <= idx < len(self.index_to_word):
            return self.index_to_word[idx]
        return "[UNK]"

    def sample_from(self, probs, temperature):
        probs = np.asarray(probs).astype("float64")
        probs = probs ** (1.0 / temperature)
        probs = probs / probs.sum()
        return np.random.choice(len(probs), p=probs), probs

    def generate(self, start_prompt, max_tokens=150, temperature=0.8, model=None):
        model = model or self.model
        start_tokens = [self.word_to_index.get(x, 1) for x in start_prompt.split()]
        sample_token = None
        while len(start_tokens) < max_tokens and sample_token != 0:
            x = np.array([start_tokens])
            y = model.predict(x, verbose=0)
            sample_token, _ = self.sample_from(y[0][-1], temperature)
            start_tokens.append(sample_token)
            start_prompt = start_prompt + " " + self._token_to_word(sample_token)
        print("\n--- LSTM generation ---\n" + start_prompt + "\n")
        return start_prompt

    def on_epoch_end(self, epoch, logs=None):
        self.generate("def ", max_tokens=80, temperature=0.9)

gen_callback_lstm = TextGeneratorLSTM(vocab)

In [ ]:
lstm_model.fit(
    train_ds,
    epochs=EPOCHS_LSTM,
    callbacks=[gen_callback_lstm],
)

## 5. Transformer (GPT) model

In [ ]:
def causal_attention_mask(batch_size, n_dest, n_src, dtype):
    i = tf.range(n_dest)[:, None]
    j = tf.range(n_src)
    m = i >= j - n_src + n_dest
    mask = tf.cast(m, dtype)
    mask = tf.reshape(mask, [1, n_dest, n_src])
    mult = tf.concat([tf.expand_dims(batch_size, -1), tf.constant([1, 1], dtype=tf.int32)], 0)
    return tf.tile(mask, mult)

class TransformerBlock(layers.Layer):
    def __init__(self, num_heads, key_dim, embed_dim, ff_dim, dropout_rate=0.1):
        super(TransformerBlock, self).__init__()
        self.attn = layers.MultiHeadAttention(num_heads, key_dim, output_shape=embed_dim)
        self.dropout_1 = layers.Dropout(dropout_rate)
        self.ln_1 = layers.LayerNormalization(epsilon=1e-6)
        self.ffn_1 = layers.Dense(ff_dim, activation="relu")
        self.ffn_2 = layers.Dense(embed_dim)
        self.dropout_2 = layers.Dropout(dropout_rate)
        self.ln_2 = layers.LayerNormalization(epsilon=1e-6)

    def call(self, inputs):
        batch_size = tf.shape(inputs)[0]
        seq_len = tf.shape(inputs)[1]
        causal_mask = causal_attention_mask(batch_size, seq_len, seq_len, tf.bool)
        attn_out, attn_scores = self.attn(
            inputs, inputs, attention_mask=causal_mask, return_attention_scores=True
        )
        out1 = self.ln_1(inputs + self.dropout_1(attn_out))
        ffn = self.ffn_2(self.ffn_1(out1))
        return self.ln_2(out1 + self.dropout_2(ffn)), attn_scores

class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, max_len, vocab_size, embed_dim):
        super(TokenAndPositionEmbedding, self).__init__()
        self.token_emb = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb = layers.Embedding(input_dim=max_len, output_dim=embed_dim)

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        return self.token_emb(x) + self.pos_emb(positions)

In [ ]:
inputs = layers.Input(shape=(None,), dtype=tf.int32)
x = TokenAndPositionEmbedding(MAX_LEN, VOCAB_SIZE, EMBEDDING_DIM)(inputs)
x, attention_scores = TransformerBlock(
    N_HEADS, KEY_DIM, EMBEDDING_DIM, FEED_FORWARD_DIM
)(x)
outputs = layers.Dense(VOCAB_SIZE, activation="softmax")(x)
gpt_model = models.Model(inputs=inputs, outputs=[outputs, attention_scores])
gpt_model.compile(
    optimizer="adam",
    loss=[losses.SparseCategoricalCrossentropy(), None],
)
gpt_model.summary()

In [ ]:
class TextGeneratorGPT(callbacks.Callback):
    def __init__(self, index_to_word):
        self.index_to_word = list(index_to_word)
        self.word_to_index = {w: i for i, w in enumerate(self.index_to_word)}

    def _token_to_word(self, idx):
        if 0 <= idx < len(self.index_to_word):
            return self.index_to_word[idx]
        return "[UNK]"

    def sample_from(self, probs, temperature):
        probs = np.asarray(probs).astype("float64")
        probs = probs ** (1.0 / temperature)
        probs = probs / probs.sum()
        return np.random.choice(len(probs), p=probs), probs

    def generate(self, start_prompt, max_tokens=150, temperature=0.8, model=None):
        model = model or self.model
        start_tokens = [self.word_to_index.get(x, 1) for x in start_prompt.split()]
        sample_token = None
        while len(start_tokens) < max_tokens and sample_token != 0:
            x = np.array([start_tokens])
            y, _ = model.predict(x, verbose=0)
            sample_token, _ = self.sample_from(y[0][-1], temperature)
            start_tokens.append(sample_token)
            start_prompt = start_prompt + " " + self._token_to_word(sample_token)
        print("\n--- GPT generation ---\n" + start_prompt + "\n")
        return start_prompt

    def on_epoch_end(self, epoch, logs=None):
        self.generate("def ", max_tokens=80, temperature=0.9)

gen_callback_gpt = TextGeneratorGPT(vocab)

In [ ]:
gpt_model.fit(
    train_ds,
    epochs=EPOCHS_GPT,
    callbacks=[gen_callback_gpt],
)

## 6. Code generation with both models

In [ ]:
# Pass model=lstm_model to generate(); do not set gen_callback_lstm.model (read-only)
print("=== LSTM ===")
gen_callback_lstm.generate("def ", max_tokens=120, temperature=0.7, model=lstm_model)
gen_callback_lstm.generate("class ", max_tokens=120, temperature=0.7, model=lstm_model)
gen_callback_lstm.generate("import ", max_tokens=80, temperature=0.8, model=lstm_model)

In [ ]:
print("=== Transformer (GPT) ===")
gen_callback_gpt.generate("def ", max_tokens=120, temperature=0.7, model=gpt_model)
gen_callback_gpt.generate("class ", max_tokens=120, temperature=0.7, model=gpt_model)
gen_callback_gpt.generate("for ", max_tokens=80, temperature=0.8, model=gpt_model)

## 7. Conclusion

- **LSTM**: good for short sequences, faster to train.
- **Transformer (GPT)**: better for long-range dependencies thanks to attention.
- Both models were trained on downloaded Python code (cpython/Lib) and can generate token sequences that resemble code (keywords, indentation, punctuation). For near-executable code quality, more data and a better-suited vocabulary (e.g. subword tokenization) would be needed.